In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Загружаем тренировочную часть датасета
df = pd.read_csv("data/train-data.csv")

# Смотрим первые 5 строк
print("Первые 5 строк датасета:")
display(df.head())

# Получаем общую информацию о датасете
print("\nИнформация о датасете:")
df.info()

# Смотрим количество пропущенных значений в каждом столбце
print("\nКоличество пропущенных значений:")
print(df.isnull().sum())

# Получаем названия всех столбцов
print("\nНазвания столбцов:")
print(df.columns.tolist())

Первые 5 строк датасета:


,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,0,Maruti Wagon R LXI CNG,Mumbai,2010,72000,CNG,Manual,First,26.6 km/kg,998 CC,58.16 bhp,5.0,NaN,1.75
1,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
2,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,18.2 kmpl,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
3,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
4,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74



Информация о датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6019 entries, 0 to 6018
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         6019 non-null   int64  
 1   Name               6019 non-null   object 
 2   Location           6019 non-null   object 
 3   Year               6019 non-null   int64  
 4   Kilometers_Driven  6019 non-null   int64  
 5   Fuel_Type          6019 non-null   object 
 6   Transmission       6019 non-null   object 
 7   Owner_Type         6019 non-null   object 
 8   Mileage            6017 non-null   object 
 9   Engine             5983 non-null   object 
 10  Power              5983 non-null   object 
 11  Seats              5977 non-null   float64
 12  New_Price          824 non-null    object 
 13  Price              6019 non-null   float64
dtypes: float64(2), int64(3), object(9)
memory usage: 658.5+ KB

Количество пропущенных значений:
Unn

In [9]:
# Безопасно удаляем столбцы (проверяем наличие перед удалением)
cols_to_drop = ['Unnamed: 0', 'New_Price', 'Name']
existing_cols = [c for c in cols_to_drop if c in df.columns]
if existing_cols:
    df = df.drop(existing_cols, axis=1)
    print(f"Удалены столбцы: {existing_cols}")

# Очистка числовых столбцов от текста (kmpl, CC, bhp)
# Проверяем тип: если столбец уже числовой — пропускаем
text_numeric_cols = ['Mileage', 'Engine', 'Power']
for col in text_numeric_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].str.extract(r'(\d+\.?\d*)').astype(float)
        print(f"Столбец '{col}': очищен от текста и преобразован в float")
    else:
        print(f"Столбец '{col}': уже числовой, очистка не требуется")

# Заполнение пропусков (пункт 4 методички)
print("\n" + "=" * 50)
print("ПРОПУСКИ ДО ЗАПОЛНЕНИЯ:")
print("=" * 50)
print(df.isnull().sum())
print(f"Всего пропусков: {df.isnull().sum().sum()}")

print("\n" + "=" * 50)
print("ЗАПОЛНЕНИЕ ПРОПУСКОВ:")
print("=" * 50)

# Числовые признаки — медианой
numeric_cols = ['Mileage', 'Engine', 'Power', 'Seats']
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        count_before = df[col].isnull().sum()
        df[col] = df[col].fillna(median_value)
        print(f"Столбец '{col}': заполнено {count_before} пропусков медианой = {median_value:.2f}")
    else:
        print(f"Столбец '{col}': пропусков нет")

# Категориальные признаки — модой
categorical_cols = ['Location', 'Fuel_Type', 'Transmission', 'Owner_Type']
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_value = df[col].mode()[0]
        count_before = df[col].isnull().sum()
        df[col] = df[col].fillna(mode_value)
        print(f"Столбец '{col}': заполнено {count_before} пропусков модой = '{mode_value}'")
    else:
        print(f"Столбец '{col}': пропусков нет")

print("\n" + "=" * 50)
print("ПРОПУСКИ ПОСЛЕ ЗАПОЛНЕНИЯ:")
print("=" * 50)
print(df.isnull().sum())
print(f"Всего пропусков: {df.isnull().sum().sum()}")

# 5. Нормализация данных (пункт 5 методички)
print("\n" + "=" * 50)
print("НОРМАЛИЗАЦИЯ ДАННЫХ (MinMaxScaler):")
print("=" * 50)

numeric_features = ['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats']
scaler = MinMaxScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

print("Нормализованные числовые признаки (первые 5 строк):")
display(df[numeric_features].head())
print("\nСтатистика после нормализации:")
print(df[numeric_features].describe().round(4))

# 6. Кодирование категориальных признаков (пункт 6 методички)
print("\n" + "=" * 50)
print("OHE-КОДИРОВАНИЕ КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ:")
print("=" * 50)

categorical_features = ['Location', 'Fuel_Type', 'Transmission', 'Owner_Type']
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

print(f"Размер до OHE: {df.shape}")
print(f"Размер после OHE: {df_encoded.shape}")
print(f"Добавлено столбцов: {df_encoded.shape[1] - df.shape[1]}")


Удалены столбцы: ['Unnamed: 0', 'New_Price', 'Name']
Столбец 'Mileage': очищен от текста и преобразован в float
Столбец 'Engine': очищен от текста и преобразован в float
Столбец 'Power': очищен от текста и преобразован в float

ПРОПУСКИ ДО ЗАПОЛНЕНИЯ:
Location               0
Year                   0
Kilometers_Driven      0
Fuel_Type              0
Transmission           0
Owner_Type             0
Mileage                2
Engine                36
Power                143
Seats                 42
Price                  0
dtype: int64
Всего пропусков: 223

ЗАПОЛНЕНИЕ ПРОПУСКОВ:
Столбец 'Mileage': заполнено 2 пропусков медианой = 18.15
Столбец 'Engine': заполнено 36 пропусков медианой = 1493.00
Столбец 'Power': заполнено 143 пропусков медианой = 97.70
Столбец 'Seats': заполнено 42 пропусков медианой = 5.00
Столбец 'Location': пропусков нет
Столбец 'Fuel_Type': пропусков нет
Столбец 'Transmission': пропусков нет
Столбец 'Owner_Type': пропусков нет

ПРОПУСКИ ПОСЛЕ ЗАПОЛНЕНИЯ:
Location     

,Year,Kilometers_Driven,Mileage,Engine,Power,Seats
0,0.571429,0.011051,0.793083,0.156261,0.045569,0.5
1,0.809524,0.006282,0.586464,0.254809,0.174971,0.5
2,0.619048,0.007051,0.542636,0.190179,0.103652,0.5
3,0.666667,0.013359,0.619261,0.198448,0.103766,0.7
4,0.714286,0.006231,0.453190,0.319946,0.202739,0.5



Статистика после нормализации:
            Year  Kilometers_Driven    Mileage     Engine      Power  \
count  6019.0000          6019.0000  6019.0000  6019.0000  6019.0000   
mean      0.7313             0.0090     0.5407     0.2613     0.1496   
std       0.1557             0.0140     0.1366     0.1012     0.1013   
min       0.0000             0.0000     0.0000     0.0000     0.0000   
25%       0.6190             0.0052     0.4523     0.1900     0.0833   
50%       0.7619             0.0081     0.5411     0.2398     0.1208   
75%       0.8571             0.0112     0.6291     0.3201     0.1975   
max       1.0000             1.0000     1.0000     1.0000     1.0000   

           Seats  
count  6019.0000  
mean      0.5277  
std       0.0806  
min       0.0000  
25%       0.5000  
50%       0.5000  
75%       0.5000  
max       1.0000  

OHE-КОДИРОВАНИЕ КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ:
Размер до OHE: (6019, 11)
Размер после OHE: (6019, 25)
Добавлено столбцов: 14


In [10]:
# Сохраняем предобработанные данные в CSV
df_encoded.to_csv("data/processed_train_data.csv", index=False)
print("Данные сохранены в data/processed_train_data.csv")
print(f"Размер: {df_encoded.shape[0]} строк, {df_encoded.shape[1]} столбцов")

Данные сохранены в data/processed_train_data.csv
Размер: 6019 строк, 25 столбцов
